# PDCH Text Modality — Fine-tuned Classifier (v3)

## 0. Config

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✅ Drive mounted")

In [ ]:
# !pip install -q torch transformers pyarrow scikit-learn
from pathlib import Path
import numpy as np, pandas as pd, json, re, time
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report

PROC_DIR  = Path('/content/drive/MyDrive/PDCH/processed_v1')   # v1 output
TURNS_PARQ= PROC_DIR / 'pdch_turns.parquet'
PAT_PARQ  = PROC_DIR / 'pdch_patients.parquet'
OUT_DIR   = PROC_DIR / 'finetune_v3'; OUT_DIR.mkdir(parents=True, exist_ok=True)

ENCODER_NAME = 'hfl/chinese-roberta-wwm-ext'
TEXT_VARIANT = 'full_dialogue'     # chosen in v2

# chunking (match v2)
MIN_CHARS = 4; MAX_CHARS = 400; MAX_TOKENS = 512

# training
FULL_FINETUNE = True               # True = all encoder params trainable (highest ceiling)
FREEZE_BELOW  = 0                  # if not full: freeze layers [0..FREEZE_BELOW)
LR = 2e-5; EPOCHS = 8; PATIENCE = 2; CHUNK_BATCH = 16
N_FOLDS = 5; INNER_VAL_FRAC = 0.15; SEED = 42
QUICK_MODE = False                 # True -> 2 folds, 3 epochs for a fast smoke test

if QUICK_MODE: N_FOLDS, EPOCHS = 2, 3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED); np.random.seed(SEED)
print('device:', DEVICE, '| full finetune:', FULL_FINETUNE, '| folds:', N_FOLDS, '| epochs:', EPOCHS)

## 1. Load data, build per-patient chunks (full_dialogue), labels

In [ ]:
turns = pd.read_parquet(TURNS_PARQ)
pat   = pd.read_parquet(PAT_PARQ)
turns = turns[turns.patient_id.isin(set(pat.patient_id))].copy()
item_cols = [c for c in pat.columns if c.startswith('hamd_') and c[5:7].isdigit()]
assert len(item_cols) == 17, len(item_cols)

def merge_consecutive(g):
    out=[]
    for role,text in zip(g.role,g.text):
        if out and out[-1][0]==role: out[-1]=(role,out[-1][1]+' '+text)
        else: out.append((role,text))
    return out
def build_full_dialogue(g):
    g=g.sort_values('turn_idx'); merged=merge_consecutive(g)
    pref={'patient':'患者：','doctor':'医生：'}
    return [pref[r]+t for r,t in merged if len(t)>=MIN_CHARS]
def pack_chunks(units, max_chars=MAX_CHARS):
    chunks,cur=[],''
    for u in units:
        if len(u)>max_chars:
            if cur: chunks.append(cur); cur=''
            for i in range(0,len(u),max_chars): chunks.append(u[i:i+max_chars])
            continue
        if len(cur)+len(u)+1<=max_chars: cur=(cur+'\n'+u) if cur else u
        else: chunks.append(cur); cur=u
    if cur: chunks.append(cur)
    return chunks if chunks else ['']

pat_index = pat.set_index('patient_id')
patient_chunks, ids = {}, []
for pid, g in turns.groupby('patient_id'):
    patient_chunks[pid] = pack_chunks(build_full_dialogue(g))
    ids.append(pid)
ids = np.array(sorted(ids))

y_sev = pat_index.loc[ids, 'sev3_code'].to_numpy().astype(int)            # (N,) 0/1/2
y_items = pat_index.loc[ids, item_cols].to_numpy().astype(float)          # (N,17) 0-4 or NaN
print('patients:', len(ids), '| sev3 dist:', np.bincount(y_sev),
      '| avg chunks/pt:', np.mean([len(patient_chunks[p]) for p in ids]).round(1))

# 3-class bands (MUST match v1)
def total_to_sev3(t):
    if t <= 7:  return 0
    if t <= 17: return 1
    return 2

## 2. Tokenizer + model (encoder + swappable head)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)

class ChunkDataset(Dataset):
    """Flattened (chunk_text, patient_row_idx) pairs for a given patient subset."""
    def __init__(self, patient_idxs):
        self.items=[]
        for pi in patient_idxs:
            pid=ids[pi]
            for ch in patient_chunks[pid]:
                self.items.append((ch, pi))
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    texts=[b[0] for b in batch]; pidx=torch.tensor([b[1] for b in batch])
    enc=tokenizer(texts, padding=True, truncation=True, max_length=MAX_TOKENS, return_tensors='pt')
    return enc, pidx

class Net(nn.Module):
    def __init__(self, target):
        super().__init__()
        self.enc = AutoModel.from_pretrained(ENCODER_NAME)
        self.target = target
        D = self.enc.config.hidden_size
        if not FULL_FINETUNE:
            for n,p in self.enc.named_parameters():
                m=re.search(r'encoder\.layer\.(\d+)\.', n)
                if m and int(m.group(1)) < FREEZE_BELOW: p.requires_grad=False
        self.drop = nn.Dropout(0.2)
        self.head = nn.Linear(D, 3) if target=='direct_severity' else nn.Linear(D, 17*5)
    def forward(self, enc):
        out = self.enc(**enc).last_hidden_state
        m = enc['attention_mask'].unsqueeze(-1).float()
        emb = (out*m).sum(1)/m.sum(1).clamp(min=1e-9)       # masked mean over chunk
        emb = self.drop(emb)
        logits = self.head(emb)
        if self.target=='multitask_items': logits = logits.view(-1,17,5)
        return logits

## 3. Train / predict for one fold

In [ ]:
def chunk_weights(patient_idxs):
    """weight each chunk by 1/n_chunks(patient) so every patient contributes equally."""
    return {pi: 1.0/len(patient_chunks[ids[pi]]) for pi in patient_idxs}

def class_weights_sev(train_idx):
    c=np.bincount(y_sev[train_idx], minlength=3).astype(float); c[c==0]=1
    w=c.sum()/(3*c); return torch.tensor(w, dtype=torch.float32, device=DEVICE)

@torch.no_grad()
def predict_patients(model, patient_idxs, target):
    model.eval()
    ds=ChunkDataset(patient_idxs); dl=DataLoader(ds, batch_size=CHUNK_BATCH, collate_fn=collate)
    if target=='direct_severity':
        acc={pi: np.zeros(3) for pi in patient_idxs}; cnt={pi:0 for pi in patient_idxs}
        for enc,pidx in dl:
            enc={k:v.to(DEVICE) for k,v in enc.items()}
            p=torch.softmax(model(enc),dim=-1).cpu().numpy()
            for j,pi in enumerate(pidx.numpy()): acc[pi]+=p[j]; cnt[pi]+=1
        return np.array([acc[pi].argmax() for pi in patient_idxs])
    else:
        acc={pi: np.zeros((17,5)) for pi in patient_idxs}
        for enc,pidx in dl:
            enc={k:v.to(DEVICE) for k,v in enc.items()}
            p=torch.softmax(model(enc),dim=-1).cpu().numpy()   # (B,17,5)
            for j,pi in enumerate(pidx.numpy()): acc[pi]+=p[j]
        preds=[]
        for pi in patient_idxs:
            item_pred=acc[pi].argmax(axis=1)          # (17,) predicted 0-4
            preds.append(total_to_sev3(item_pred.sum()))   # sum all 17 -> bin
        return np.array(preds)

def train_fold(train_idx, target):
    # inner val for early stopping (patient-level, stratified)
    tr, va = train_test_split(train_idx, test_size=INNER_VAL_FRAC,
                              stratify=y_sev[train_idx], random_state=SEED)
    model=Net(target).to(DEVICE)
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    cw=class_weights_sev(tr); ce_sev=nn.CrossEntropyLoss(weight=cw)
    ce_item=nn.CrossEntropyLoss(reduction='none')
    wmap=chunk_weights(tr)
    ds=ChunkDataset(tr); dl=DataLoader(ds,batch_size=CHUNK_BATCH,shuffle=True,collate_fn=collate)
    best_f1, best_state, bad = -1, None, 0
    for ep in range(EPOCHS):
        model.train()
        for enc,pidx in dl:
            enc={k:v.to(DEVICE) for k,v in enc.items()}
            logits=model(enc); w=torch.tensor([wmap[int(pi)] for pi in pidx],dtype=torch.float32,device=DEVICE)
            if target=='direct_severity':
                yb=torch.tensor(y_sev[pidx.numpy()],device=DEVICE)
                loss=(ce_sev(logits,yb)*w).sum()/w.sum()
            else:
                yi=torch.tensor(y_items[pidx.numpy()],device=DEVICE)        # (B,17) NaN allowed
                mask=~torch.isnan(yi)
                yi_f=torch.nan_to_num(yi,nan=0.0).long()
                per=ce_item(logits.reshape(-1,5), yi_f.reshape(-1)).view(-1,17)
                per=per*mask.float()
                loss=((per.sum(1)/mask.float().sum(1).clamp(min=1))*w).sum()/w.sum()
            opt.zero_grad(); loss.backward(); opt.step()
        vp=predict_patients(model, va, target); vf=f1_score(y_sev[va], vp, average='macro')
        if vf>best_f1: best_f1, best_state, bad = vf, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 0
        else:
            bad+=1
            if bad>=PATIENCE: break
    model.load_state_dict(best_state)
    return model

## 4. Run both targets across the SAME folds

In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(np.arange(len(ids)), y_sev))
# leakage guard
for k,(tr,te) in enumerate(folds):
    assert set(ids[tr]).isdisjoint(set(ids[te])), f'LEAKAGE fold {k}'
print('folds built, train/test patient-disjoint verified')

results = {'direct_severity': [], 'multitask_items': []}
oof = {t: np.full(len(ids), -1) for t in results}
for target in results:
    print(f'\n===== {target} =====')
    for k,(tr,te) in enumerate(folds):
        t0=time.time(); model=train_fold(tr, target)
        pred=predict_patients(model, te, target)
        oof[target][te]=pred
        f1=f1_score(y_sev[te], pred, average='macro')
        results[target].append(f1)
        print(f'  fold {k+1}/{N_FOLDS}  macro-F1={f1:.3f}  ({time.time()-t0:.0f}s)')
        del model; torch.cuda.empty_cache() if DEVICE=='cuda' else None

## 5. Results — fine-tuned vs probe, and target comparison

In [ ]:
print('3-class severity, patient-level 5-fold CV (macro-F1)\n')
print(f"{'method':<34}{'mean':>8}{'std':>8}")
print(f"{'v2 frozen probe (logreg)':<34}{0.449:>8.3f}{'  --':>8}")
for t in results:
    arr=np.array(results[t]); print(f"{'v3 finetune · '+t:<34}{arr.mean():>8.3f}{arr.std():>8.3f}")

print('\nper-fold:')
for t in results: print(f'  {t:<20}', [round(x,3) for x in results[t]])

print('\nOOF classification report (best target):')
best_t=max(results, key=lambda t: np.mean(results[t]))
print('best target =', best_t)
print(classification_report(y_sev, oof[best_t],
      target_names=['none_minimal','mild_moderate','severe'], digits=3, zero_division=0))

## 6. Save metrics

In [ ]:
summary = {
    'encoder': ENCODER_NAME, 'text_variant': TEXT_VARIANT,
    'full_finetune': FULL_FINETUNE, 'n_folds': N_FOLDS, 'epochs': EPOCHS, 'lr': LR,
    'probe_baseline_logreg_3class': 0.449,
    'results_macroF1': {t: {'per_fold': [float(x) for x in results[t]],
                            'mean': float(np.mean(results[t])),
                            'std': float(np.std(results[t]))} for t in results},
    'best_target': best_t,
}
json.dump(summary, open(OUT_DIR/'v3_results.json','w'), indent=2, ensure_ascii=False)
np.savez_compressed(OUT_DIR/'v3_oof_predictions.npz', ids=ids, y_true=y_sev,
                    **{f'oof_{t}': oof[t] for t in results})
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('\nsaved to', OUT_DIR)